<a href="https://colab.research.google.com/github/Fastpacer/Abstract_ART_Transformer/blob/main/ART_TRANSFORM_EFFECTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install required packages
!pip install torch torchvision opencv-python pillow numpy tqdm gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.8/319.8 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141.9 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/1

In [3]:
# Import and check versions
import torch
import cv2
import numpy as np
print(f"Torch version: {torch.__version__}")
print(f"OpenCV version: {cv2.__version__}")
print(f"Running on CPU: {torch.device('cpu')}")

Torch version: 2.4.1+cu121
OpenCV version: 4.10.0
Running on CPU: cpu


In [4]:
# Optional: Limit torch threads for better Colab compatibility
import torch
torch.set_num_threads(2)

In [5]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import numpy as np
from PIL import Image
import cv2
import gradio as gr
from tqdm import tqdm
import tempfile
import os

class LightweightStyleTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        # Simplified architecture for CPU
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(inplace=True)
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 3, 3, stride=1, padding=1),
            nn.Tanh()
        )

    def forward(self, x):
        x = self.encoder(x)
        return self.decoder(x)

class CPUArtTransformer:
    def __init__(self):
        self.device = 'cpu'
        self.model = LightweightStyleTransformer().to(self.device)
        self.style_effects = {
            "Cubism": self.cubism_effect,
            "Impressionism": self.impressionism_effect,
            "Abstract": self.abstract_effect,
            "Pop Art": self.pop_art_effect
        }

    def preprocess_image(self, image):
        if isinstance(image, np.ndarray):
            image = Image.fromarray(image)
        transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
        ])
        return transform(image).unsqueeze(0)  # Add batch dimension

    def postprocess_image(self, tensor):
        tensor = tensor.detach().cpu().numpy()  # Detach and move tensor to CPU
        return np.clip(tensor.transpose(1, 2, 0), 0, 1)  # Clip values and transpose

    def apply_brushstroke(self, image, intensity):
        d = int(5 + intensity * 15)
        sigmaColor = int(10 + intensity * 200)
        sigmaSpace = int(10 + intensity * 200)
        return cv2.bilateralFilter(image, d, sigmaColor, sigmaSpace)

    def apply_color_quantization(self, image, n_colors):
        h, w = image.shape[:2]
        image = image.reshape(-1, 3)
        image = np.float32(image)

        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
        _, labels, centers = cv2.kmeans(image, n_colors, None, criteria, 10, cv2.KMEANS_PP_CENTERS)

        centers = np.uint8(centers)
        quantized = centers[labels.flatten()]
        return quantized.reshape(h, w, 3)

    def apply_geometric_distortion(self, image, intensity):
        h, w = image.shape[:2]
        map_x = np.zeros((h, w), np.float32)
        map_y = np.zeros((h, w), np.float32)

        for y in range(h):
            for x in range(w):
                map_x[y, x] = x + intensity * 10 * np.sin(y/30.0)
                map_y[y, x] = y + intensity * 10 * np.cos(x/30.0)

        return cv2.remap(image, map_x, map_y, cv2.INTER_LINEAR)

    def cubism_effect(self, image, intensity):
        # Simplified cubism effect
        triangles = self.apply_geometric_distortion(image, intensity)
        color_reduced = self.apply_color_quantization(triangles, n_colors=4 + int(intensity * 4))
        return self.apply_brushstroke(color_reduced, intensity * 0.5)

    def impressionism_effect(self, image, intensity):
        # Impressionistic effect with brushstrokes
        blurred = cv2.GaussianBlur(image, (0, 0), intensity * 5)
        brushstroke = self.apply_brushstroke(blurred, intensity)
        color_enhanced = cv2.addWeighted(brushstroke, 1 + intensity, brushstroke, 0, 0)
        return color_enhanced

    def abstract_effect(self, image, intensity):
        # Abstract effect with geometric distortion
        distorted = self.apply_geometric_distortion(image, intensity)
        color_reduced = self.apply_color_quantization(distorted, n_colors=3 + int(intensity * 5))
        return cv2.GaussianBlur(color_reduced, (0, 0), intensity * 3)

    def pop_art_effect(self, image, intensity):
        # Pop art effect with vibrant colors
        color_reduced = self.apply_color_quantization(image, n_colors=2 + int(intensity * 4))
        enhanced = cv2.convertScaleAbs(color_reduced, alpha=1 + intensity, beta=intensity * 25)
        return enhanced

    def generate_video_frames(self, image, style_name, num_frames=90):
        frames = []
        base_image = np.array(image)

        for i in tqdm(range(num_frames), desc=f"Generating {style_name} video"):
            intensity = i / num_frames
            styled_frame = self.style_effects[style_name](base_image.copy(), intensity)
            frames.append(styled_frame)

        return frames

    def create_video(self, frames, output_path, fps=24):
        height, width, _ = frames[0].shape
        fourcc = cv2.VideoWriter_fourcc(*'VP80')  # Use WebM for compatibility
        video_writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

        for frame in frames:
            video_writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

        video_writer.release()

class ArtTransformerInterface:
    def __init__(self):
        self.transformer = CPUArtTransformer()

    def process_image(self, input_image, style):
        if input_image is None:
            return None

        frames = self.transformer.generate_video_frames(input_image, style)

        with tempfile.NamedTemporaryFile(delete=False, suffix='.webm') as tmp_file:
            output_path = tmp_file.name

        self.transformer.create_video(frames, output_path)
        return output_path

def create_gradio_interface():
    interface = ArtTransformerInterface()

    return gr.Interface(
        fn=interface.process_image,
        inputs=[
            gr.Image(type="pil", label="Input Image"),
            gr.Dropdown(
                choices=list(interface.transformer.style_effects.keys()),
                label="Art Style",
                value="Cubism"
            )
        ],
        outputs=gr.Video(label="Generated Abstract Art Video"),
        title="CPU-Friendly Abstract Art Transformer",
        description="""Transform images into abstract art videos using various artistic styles.
                       This version is optimized for CPU usage. Processing may take a few minutes."""
    )

if __name__ == "__main__":
    demo = create_gradio_interface()
    demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ab7d12e25af7bfc85d.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
